# Семинар 2. Инструменты и проверяемые контракты

**Python 3.10+ · только стандартная библиотека**

Это автономный учебный ноутбук: загрузите файл в Colab либо Jupyter и выполняйте ячейки сверху вниз.
Установка пакетов, ключи, GPU и сетевой доступ не требуются. Здесь используется **офлайн-симуляция**, а не настоящая LLM.

Предпосылки: функции, словари, исключения и базовые тесты `assert`; материал предыдущих занятий.
Четыре ячейки с тегом `exercise` содержат задания. После заполнения каждого задания запускайте его открытые проверки.
Пока функция не реализована, её вызов завершится `NotImplementedError`.

Баллы: 2 за каждое задание, 2 за корректный эксперимент и разбор ограничений.

Критерий сдачи: четыре функции, пройденные проверки, один дополнительный отрицательный случай и ответы в итоговом отчёте.
Результаты симуляции нельзя подписывать как качество или скорость языковой модели.


Вы построите реестр двух операций, защитите чтение учебной папки и отделите каталог от исполняющих функций.
    `inputSchema` в каталоге похожа на описание MCP-инструмента, но **этот ноутбук не реализует MCP**:
    нет клиента/сервера, транспорта, discovery и протокольных метаданных.
    Схемы здесь поддерживают только объект с обязательными строковыми/целочисленными полями и запретом лишних ключей.
    Пример с официальным SDK находится в дополнении `lesson-02/mcp-demo`.

    Источники: [MCP tools 2026-07-28](https://modelcontextprotocol.io/specification/2026-07-28/server/tools),
    [Toolformer](https://arxiv.org/abs/2302.04761).


In [1]:
import copy
import json
import tempfile
from pathlib import Path

class ContractError(ValueError):
    pass

def expect_error(exception, thunk):
    try:
        thunk()
    except exception:
        return
    raise AssertionError(f"Expected {exception.__name__}")

WORK = tempfile.TemporaryDirectory(prefix="agents-lesson02-")
ROOT_DIR = Path(WORK.name) / "allowed"
ROOT_DIR.mkdir()
(ROOT_DIR / "schedule.json").write_text(json.dumps({"room": "314", "time": None}), encoding="utf-8")
(ROOT_DIR / "broken.json").write_text("not json", encoding="utf-8")
(Path(WORK.name) / "outside.json").write_text('{"private": true}', encoding="utf-8")

ADD_SCHEMA = {"type": "object", "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
              "required": ["a", "b"], "additionalProperties": False}
READ_SCHEMA = {"type": "object", "properties": {"path": {"type": "string"}},
               "required": ["path"], "additionalProperties": False}
print("OFFLINE: учебные файлы созданы во временной папке")


OFFLINE: учебные файлы созданы во временной папке


### Решённый пример: наличие и пустое значение
    Не смешивайте `null` внутри корректного документа с отсутствием поля и с ошибкой чтения.


In [2]:
def field_observation(document, key):
    return {"found": key in document, "value": document.get(key)}

assert field_observation({"time": None}, "time") == {"found": True, "value": None}
assert field_observation({}, "time") == {"found": False, "value": None}
print(field_observation({"time": None}, "time"))


{'found': True, 'value': None}


## Задание 1. Строгая проверка аргументов — 2 балла

Реализуйте `validate_args(schema, args)`.
    Предусловие: `schema` — доверенная схема из поддерживаемого подмножества выше.
    `args` должен быть именно `dict`; все required-поля присутствуют; поля вне properties запрещены.
    Для integer принимается именно `int`, **не bool**; для string — `str`.
    Возвращается независимая копия аргументов. Любое нарушение входа вызывает `ContractError`.
    Схему и исходные аргументы не изменяйте. Никакой автоматической конвертации строк в числа.


In [3]:
def validate_args(schema, args):
    if type(args) is not dict:
        raise ContractError("Аргументы должны быть словарем (dict)")

    properties = schema.get("properties", {})
    required = schema.get("required", [])

    for req in required:
        if req not in args:
            raise ContractError(f"Отсутствует обязательное поле: {req}")

    for key, value in args.items():
        if key not in properties:
            raise ContractError(f"Передано неизвестное поле: {key}")

        expected_type = properties[key].get("type")

        if expected_type == "string":
            if type(value) is not str:
                raise ContractError(f"Поле '{key}' должно быть строкой (str)")

        elif expected_type == "integer":
            if type(value) is not int:
                raise ContractError(f"Поле '{key}' должно быть целым числом (int), не bool")

    return args.copy()

In [4]:
assert validate_args(ADD_SCHEMA, {"a": 7, "b": 5}) == {"a": 7, "b": 5}
for bad in ({"a": True, "b": 2}, {"a": "1", "b": 2}, {"a": 1}, {"a": 1, "b": 2, "extra": 0}, []):
    expect_error(ContractError, lambda bad=bad: validate_args(ADD_SCHEMA, bad))
original = {"a": 1, "b": 2}
validated = validate_args(ADD_SCHEMA, original)
validated["a"] = 99
assert original["a"] == 1
print("PASS: validate_args")


PASS: validate_args


## Задание 2. Чтение в заданной папке — 2 балла

`read_json_document(root, relative_path)` читает UTF-8 JSON только внутри `root`.
    Путь — непустая строка, не абсолютный. После `resolve` он обязан оставаться внутри `root.resolve()`.
    Разрешено расширение `.json`; нарушения пути вызывают `PermissionError`.
    Для отсутствующего файла и повреждённого JSON сохраняйте стандартные исключения.
    Это контролируемая учебная папка; проверка пути **не является полной OS-песочницей** и не закрывает гонку изменения ссылок другим процессом.


In [5]:
def read_json_document(root, relative_path):
    if not relative_path:
        raise PermissionError()

    rel_path = Path(relative_path)

    if rel_path.is_absolute():
        raise PermissionError()

    if rel_path.suffix != ".json":
        raise PermissionError()

    root_resolved = Path(root).resolve()
    target_resolved = (root_resolved / rel_path).resolve()

    try:
        target_resolved.relative_to(root_resolved)
    except ValueError:
        raise PermissionError()

    if target_resolved == root_resolved:
        raise PermissionError()

    with open(target_resolved, "r", encoding="utf-8") as f:
        return json.load(f)

In [6]:
assert read_json_document(ROOT_DIR, "schedule.json")["room"] == "314"
expect_error(PermissionError, lambda: read_json_document(ROOT_DIR, "../outside.json"))
expect_error(PermissionError, lambda: read_json_document(ROOT_DIR, str(ROOT_DIR / "schedule.json")))
expect_error(PermissionError, lambda: read_json_document(ROOT_DIR, "file.txt"))
expect_error(FileNotFoundError, lambda: read_json_document(ROOT_DIR, "missing.json"))
expect_error(json.JSONDecodeError, lambda: read_json_document(ROOT_DIR, "broken.json"))
link = ROOT_DIR / "link.json"
try:
    link.symlink_to(Path(WORK.name) / "outside.json")
except (OSError, NotImplementedError):
    print("SKIP: symlink creation unavailable on this platform")
else:
    expect_error(PermissionError, lambda: read_json_document(ROOT_DIR, "link.json"))
    link.unlink()
print("PASS: read_json_document")


PASS: read_json_document


In [7]:
CALLS = []
def add_handler(a, b):
    CALLS.append("math.add")
    return a + b

def read_handler(path):
    CALLS.append("docs.read")
    return read_json_document(ROOT_DIR, path)

REGISTRY = {
    "math.add": {"description": "Сумма двух целых чисел; без преобразования строк", "inputSchema": ADD_SCHEMA, "handler": add_handler},
    "docs.read": {"description": "Прочитать JSON из учебной папки по относительному пути", "inputSchema": READ_SCHEMA, "handler": read_handler},
}


## Задание 3. Каталог без функций — 2 балла

`make_catalog(registry)` возвращает список по возрастанию имени.
    Каждый элемент содержит **ровно** name, description, inputSchema. Функции handler не включаются.
    Вложенная схема копируется независимо. Каталог должен сериализоваться через `json.dumps`.
    Форма напоминает описание MCP tool, но это обычная локальная структура без протокола.


In [8]:
def make_catalog(registry):
    catalog = []
    for key, tool in registry.items():
        catalog.append({
            "name": tool.get("name", key),
            "description": tool["description"],
            "inputSchema": copy.deepcopy(tool["inputSchema"])
        })

    return sorted(catalog, key=lambda item: item["name"])

In [9]:
catalog = make_catalog(REGISTRY)
assert [tool["name"] for tool in catalog] == ["docs.read", "math.add"]
assert all(set(tool) == {"name", "description", "inputSchema"} for tool in catalog)
json.dumps(catalog)
catalog[0]["inputSchema"]["properties"]["path"]["type"] = "integer"
assert READ_SCHEMA["properties"]["path"]["type"] == "string"
print("PASS: make_catalog")


PASS: make_catalog


## Задание 4. Допуск до исполнения — 2 балла

`dispatch_tool(name, args, registry, allowed)` возвращает `{"ok": True, "value": ...}`
    либо `{"ok": False, "error": code}`. Порядок проверок: известность имени → allowed → схема → handler.
    Коды: unknown_tool, forbidden, invalid_arguments, tool_error. Нестроковое имя тоже unknown_tool.
    При ошибке схемы handler не вызывается. Любое обычное исключение обработчика даёт tool_error;
    текст исключения наружу не возвращайте. KeyboardInterrupt/SystemExit не перехватывайте.


In [12]:
def dispatch_tool(name, args, registry, allowed):
    if type(name) is not str or name not in registry:
        return {"ok": False, "error": "unknown_tool"}

    if name not in allowed:
        return {"ok": False, "error": "forbidden"}

    try:
        valid_args = validate_args(registry[name]["inputSchema"], args)
    except Exception:
        return {"ok": False, "error": "invalid_arguments"}

    try:
        value = registry[name]["handler"](**valid_args)
        return {"ok": True, "value": value}
    except Exception:
        return {"ok": False, "error": "tool_error"}

In [13]:
CALLS.clear()
allowed = set(REGISTRY)
assert dispatch_tool("math.add", {"a": 3, "b": 4}, REGISTRY, allowed) == {"ok": True, "value": 7}
assert CALLS == ["math.add"]
assert dispatch_tool("math.add", {"a": True, "b": 4}, REGISTRY, allowed)["error"] == "invalid_arguments"
assert dispatch_tool("docs.read", {"path": "schedule.json"}, REGISTRY, set())["error"] == "forbidden"
assert dispatch_tool("missing", {}, REGISTRY, allowed)["error"] == "unknown_tool"
assert CALLS == ["math.add"], "Rejected requests must not execute"
assert dispatch_tool("docs.read", {"path": "broken.json"}, REGISTRY, allowed)["error"] == "tool_error"
print("PASS: dispatch_tool")


PASS: dispatch_tool


## Эксперимент
    Это фиксированный набор запросов, **не сгенерированный LLM**. Сначала предскажите итог каждого случая.
    Затем выполните таблицу и добавьте собственный отрицательный случай.
    Число исполнений включает обработчик, завершившийся с ошибкой; отклонённые до handler запросы не считаются исполненными.


In [16]:
cases = [
    ("valid add", "math.add", {"a": 2, "b": 8}, set(REGISTRY)),
    ("wrong type", "math.add", {"a": "2", "b": 8}, set(REGISTRY)),
    ("forbidden read", "docs.read", {"path": "schedule.json"}, {"math.add"}),
    ("missing file", "docs.read", {"path": "absent.json"}, set(REGISTRY)),
    ("valid read", "docs.read", {"path": "schedule.json"}, set(REGISTRY)),
    ("unknown tool", "docs.write", {"path": "test.json", "content": "123"}, set(REGISTRY)),
    ("not string name", 404, {"a": 1}, set(REGISTRY)),
    ("path traversal", "docs.read", {"path": "../config.json"}, set(REGISTRY)),
    ("wrong extension", "docs.read", {"path": "schedule.txt"}, set(REGISTRY)),
    ("missing required arg", "math.add", {"a": 2}, set(REGISTRY))
]
CALLS.clear()
rows = []
for label, name, args, allowed in cases:
    before = len(CALLS)
    result = dispatch_tool(name, args, REGISTRY, allowed)
    rows.append({"case": label, "status": "ok" if result["ok"] else result["error"], "executed": len(CALLS) > before})
print(json.dumps(rows, ensure_ascii=False, indent=2))
assert sum(row["executed"] for row in rows) == 5
assert sum(row["status"] == "ok" for row in rows) == 2


[
  {
    "case": "valid add",
    "status": "ok",
    "executed": true
  },
  {
    "case": "wrong type",
    "status": "invalid_arguments",
    "executed": false
  },
  {
    "case": "forbidden read",
    "status": "forbidden",
    "executed": false
  },
  {
    "case": "missing file",
    "status": "tool_error",
    "executed": true
  },
  {
    "case": "valid read",
    "status": "ok",
    "executed": true
  },
  {
    "case": "unknown tool",
    "status": "unknown_tool",
    "executed": false
  },
  {
    "case": "not string name",
    "status": "unknown_tool",
    "executed": false
  },
  {
    "case": "path traversal",
    "status": "tool_error",
    "executed": true
  },
  {
    "case": "wrong extension",
    "status": "tool_error",
    "executed": true
  },
  {
    "case": "missing required arg",
    "status": "invalid_arguments",
    "executed": false
  }
]


## Отчёт
    1. Чем отличаются invalid_arguments и tool_error с точки зрения факта запуска?
    2. Почему три исполненных операции дали только два успешных результата?
    3. Что нужно добавить, чтобы каталог стал частью реальной интеграции MCP?
    4. Какой дополнительный тест вы добавили и какую ошибку он ловит?

    Запишите наблюдаемые статусы и ограничения проверки пути. Не делайте вывод о качестве LLM: её здесь не вызывали.
    Дополнительно: опишите навык проверки расписания, использующий docs.read, и укажите условие его применимости.


При invalid_arguments инструмент даже не начинает работу из-за неверного формата данных. При tool_error инструмент успешно запускается, но ломается в процессе выполнения задачи.

Из трёх запущенных инструментов один завершился сбоем (попытался прочитать несуществующий файл), поэтому успешных результатов только два.

Необходим протокол связи (например, JSON-RPC), чтобы внешние программы могли запрашивать список инструментов и вызывать их по стандарту.

Добавлены тесты на неизвестное имя инструмента (unknown_tool), пропуск обязательного аргумента (invalid_arguments), попытку выйти за пределы папки и неверное расширение файла (оба вызывают tool_error).

Наблюдаемые статусы и ограничения
Зафиксированы статусы: ok, invalid_arguments, forbidden, tool_error и unknown_tool.
Проверка пути запрещает чтение абсолютных путей, чужих форматов (не .json) и выход на уровень выше. Однако это не строгая системная песочница: она не спасет, если другой процесс подменит файл прямо в момент чтения.

Навык проверки расписания
Для работы с расписанием используется инструмент docs.read, который извлекает данные из файла schedule.json. Это применимо только в том случае, если инструмент docs.read явно добавлен в список разрешённых (allowed), а файл с корректным текстом существует в целевой папке.

In [ ]:
WORK.cleanup()
print("Временные учебные файлы удалены")
